# Fleet Semantic v0.7.1 — Single Model — Serious Bank + Automatic Hub Release

**Before running:** rotate any Hugging Face token that has ever been pasted into a chat,
notebook source, or public place.

In Colab, open the **Secrets** panel (key icon), create a secret named `HF_TOKEN`,
and grant this notebook access to it. Use a Hugging Face **write** token that can write
to `noumenon-labs`.

Upload `fleet-semantic-v0.7.zip` when Cell 1 asks.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!rm -rf /content/fleet-semantic-v0.7
!mkdir -p /content/fleet-semantic-v0.7
!unzip -q /content/fleet-semantic-v0.7.zip -d /content/fleet-semantic-v0.7
%cd /content/fleet-semantic-v0.7
!pip install -q -U pip
!pip install -q -e ".[server]"

In [ ]:
import os
from google.colab import userdata

# Token stays in process memory. It is not written into Fleet files.
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
!fleet build \
  --config configs/experts.json \
  --output artifacts/fleet-bank

In [ ]:
!fleet semantic-build \
  --bank artifacts/fleet-bank \
  --anchors configs/semantic_anchors.json

In [ ]:
!fleet validate artifacts/fleet-bank

## Automatic release

The next cell does all of this itself:

1. reruns the routing benchmark on every active serious expert;
2. requires the NULL/base probe to work;
3. rejects non-redistributable or unpinned experts;
4. constructs a clean **model-only** release folder;
5. creates/updates the Hub model repo;
6. uploads it.

If the routing benchmark fails, **nothing is uploaded**.

In [ ]:
!fleet release \
  --bank artifacts/fleet-bank \
  --repo noumenon-labs/Fleet-Semantic-4B \
  --num-workers 4